# 02 - Wheel driver validation on the MuJoCo TurtleBot3

Checks that the driver (`drivers/mujoco_tb3.py`) turns a desired wheel speed into that wheel speed despite the actuator dead zone and offset, and measures how well encoder odometry matches the true motion. Ground truth is used here for evaluation only.

In [ ]:
!pip install -q mujoco
!test -d robot || git clone -q https://github.com/hongjiezheng-1-hue/robot.git
!cd robot && git pull -q
!test -d robotis_mujoco_menagerie || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ROBOTIS-GIT/robotis_mujoco_menagerie.git
!cd robotis_mujoco_menagerie && git sparse-checkout set robotis_tb3

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import sys
sys.path.insert(0, "robot")
import numpy as np
from drivers.mujoco_tb3 import MujocoTB3Driver

SCENE = "robotis_mujoco_menagerie/robotis_tb3/scene_turtlebot3_waffle_pi.xml"
WHEEL_RADIUS = 0.033   # m
TRACK_NOMINAL = 0.288  # m, XML wheel separation of the Waffle Pi

drv = MujocoTB3Driver(SCENE)
print("control period", drv.control_dt, "s,", drv.substeps, "physics steps per control period")

In [ ]:
def run_constant(drv, wl, wr, seconds=8.0):
    drv.reset()
    rec = []
    for _ in range(int(seconds / drv.control_dt)):
        enc = drv.step(wl, wr)
        x, y, yaw = drv.true_pose()   # ground truth: evaluation only
        rec.append([enc.time, enc.omega[0], enc.omega[1], x, y, yaw])
    return np.array(rec)

def steady_stats(rec, window=1.0):
    n = int(window / (rec[1, 0] - rec[0, 0]))
    s = rec[-n:]
    dt = s[-1, 0] - s[0, 0]
    heading = s[:, 5].mean()
    v = ((s[-1, 3] - s[0, 3]) * np.cos(heading) + (s[-1, 4] - s[0, 4]) * np.sin(heading)) / dt   # signed true forward speed
    wz = (np.unwrap(s[:, 5])[-1] - np.unwrap(s[:, 5])[0]) / dt
    return v, wz, s[:, 1].mean(), s[:, 2].mean()

print("straight: desired | measured wheel L, R | true v | odometry v | true v / odometry v")
for des in (0.5, 1.0, 2.0, 3.0, 5.0):
    v, wz, wl, wr = steady_stats(run_constant(drv, des, des))
    v_odo = WHEEL_RADIUS * (wl + wr) / 2
    print(f"          {des:5.2f}   | {wl:6.2f} {wr:6.2f}      | {v:+.3f} | {v_odo:.3f}      | {v / v_odo:.2f}")

print("spin:     desired | measured wheel L, R | true yaw rate | odometry yaw rate (nominal track) | effective track")
for des in (0.5, 1.0, 2.0):
    v, wz, wl, wr = steady_stats(run_constant(drv, -des, des))
    w_odo = WHEEL_RADIUS * (wr - wl) / TRACK_NOMINAL
    print(f"          {des:5.2f}   | {wl:6.2f} {wr:6.2f}      | {wz:+.3f}       | {w_odo:+.3f}                           | {WHEEL_RADIUS * (wr - wl) / wz:.3f} m")

## What to look for

- Measured wheel speeds should be close to the desired ones, including at low desired speeds that the raw actuator could not reach (dead zone).
- The ratio of true to odometry forward speed and the effective track width are the odometry calibration quantities for the EKF motion model.